In [1]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import sys

sys.path.append("../..")

In [2]:
from src.data.load_data import load_data
from src.models.kernel_regression import KernelRegression
from src.conformal_prediction.stable_cp import StableConformalPredictor

Load data

In [3]:
input_points, output_points = load_data("friedman1")

In [4]:
train_input_points, test_input_points, train_output_points, test_output_points = (
    train_test_split(input_points, output_points, random_state=0)
)

Instantiate predictor

In [5]:
# loss_name = "log_cosh"
# loss_params = {"alpha":1.}

loss_name = "pseudo_huber"
loss_params = {"alpha": 1.0}

# loss_name = "smoothed_pinball"
# loss_params = {"alpha":1., "tau":0.5}

In [6]:
predictor = KernelRegression(
    solver="lbfgs", loss_name=loss_name, loss_params=loss_params, lam=0.5
)

Instantiate region predictor

In [7]:
conformal_predictor = StableConformalPredictor(
    predictor, non_conformity_name="absolute"
)
region_predictor = conformal_predictor.fit_predict(
    train_input_points, train_output_points, test_input_points
)

In [8]:
confidence_control_level = 0.1
prediction_regions = region_predictor(confidence_control_level)

In [9]:
prediction_regions

[{'upper': [-1.663501539765083,1.3965657783508827],
  'lower': [-0.841831124788954,0.7406027907439005]},
 {'upper': [-0.9260001387041322,2.1696954416878285],
  'lower': [-0.5360688097913747,1.0289634786673179]},
 {'upper': [-1.6360848037026576,1.5165025735979223],
  'lower': [-0.7864660922898999,0.7445878376332399]},
 {'upper': [-1.7274012000280428,1.4181270777358865],
  'lower': [-0.82143030885766,0.7125832184599625]},
 {'upper': [-1.9831230927984917,1.1992872499571599],
  'lower': [-0.8994861451254225,0.6249416897230797]},
 {'upper': [-1.4746855565263075,1.6067889020712418],
  'lower': [-0.7592899193689884,0.8086567753734231]},
 {'upper': [-1.3657490568011683,1.6601891754570308],
  'lower': [-0.7402967200277236,0.8564936459486465]},
 {'upper': [-0.8906901723139898,2.2357579398546354],
  'lower': [-0.5130312809075337,1.036196001657136]},
 {'upper': [-1.939556647352194,1.1512403676625567],
  'lower': [-0.925323424588882,0.6271780811355302]},
 {'upper': [-2.2236190574828343,0.7736635068

In [9]:
coverage_upper = np.mean(
    [
        test_output_point in prediction_region["upper"]
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage_upper)

test coverage:  0.992


In [10]:
coverage_lower = np.mean(
    [
        test_output_point in prediction_region["lower"]
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage_lower)

test coverage:  0.632
